# Haltere campaniform sensilla → wing steering motor neurons

Do **b1, b2, tp1, tp2 and tpn** receive similar input from haltere campaniform
sensilla (HCS)?  If HCS encode different flight variables (yaw / pitch / roll
angular velocity), then a motor neuron whose input pattern resembles a
well-characterised one is plausibly recruited under similar conditions.

Run on **three connectomes independently**, using **direct (monosynaptic) connections
only**:

| dataset | animal | source | hemispheres |
|---|---|---|---|
| `male-cns:v1.0` | male | neuPrint | L and R |
| `manc:v1.2.3` | male | neuPrint | L and R |
| `fanc:v604` | **female** | local tables, converted | **L only** |

---
### A note on how this notebook is ordered

Sections 1–5 are **blind**: they treat all five motor neurons identically and
never group them. The reference-vs-query question ("which of tp1/tp2/tpn looks
most like b1/b2?") is answered only in **Section 6**, after the matrices and
controls already exist.  Every parameter was fixed in `hcs_lib.py` before any
result was inspected.  If you change one after reading Section 6, record the
change and why — otherwise the analysis stops being blind.

## 1 · Setup

Everything reads cached tables, so no neuPrint token is needed.  A token is required
only to refresh the neuPrint cache (`--refresh`).  FANC comes from a local conversion
(`tools/fanc_to_cache.py`) rather than a query, and is not distributed with the repo —
if its cache is absent the notebook simply runs on the two neuPrint datasets.

In [1]:
import os, numpy as np, pandas as pd
from scipy.stats import pearsonr
import hcs_lib as H, hcs_plots as P

pd.set_option("display.width", 200)

B = {}
for name in ("male-cns", "manc", "fanc"):
    try:
        B[name] = H.load(name)
    except RuntimeError as e:                     # FANC cache not built on this machine
        print(f"  skipping {name}: {str(e).splitlines()[0]}")
SIDES = {n: H.DATASETS[n]["sides"] for n in B}
B

  loaded male-cns from cache: <Bundle male-cns (male-cns:v1.0): 339 HCS, 10 MN, 452 IN, 163 direct edges, 975 IN->MN edges>
  loaded manc from cache: <Bundle manc (manc:v1.2.3): 333 HCS, 10 MN, 555 IN, 201 direct edges, 1227 IN->MN edges>
  loaded fanc from cache: <Bundle fanc (fanc:v604): 123 HCS, 5 MN, 693 IN, 78 direct edges, 1025 IN->MN edges>


{'male-cns': <Bundle male-cns (male-cns:v1.0): 339 HCS, 10 MN, 452 IN, 163 direct edges, 975 IN->MN edges>,
 'manc': <Bundle manc (manc:v1.2.3): 333 HCS, 10 MN, 555 IN, 201 direct edges, 1227 IN->MN edges>,
 'fanc': <Bundle fanc (fanc:v604): 123 HCS, 5 MN, 693 IN, 78 direct edges, 1025 IN->MN edges>}

### What is in the neuron sets

**HCS** = everything entering on the haltere nerve (`DMetaN`) whose type is in the
`SApp` series.  339 cells in male-cns, 333 in MANC — against ~340 campaniform
sensilla described anatomically on the haltere, which is the main evidence the
definition is right.  Roughly 45% are the generic `SApp` type with no field
assignment; that limits how far the *which sensory variable* question can go.

FANC's equivalent is 123 afferents typed `haltere_L`, with **no field subtypes at all**
and only one hemisphere reconstructed.  Its absence of a connection is therefore weaker
evidence than its presence.

In [3]:
for n, b in B.items():
    named = (~b.hcs.type.eq("SApp")).sum() if n != "fanc" else 0
    print(f"{n:9s} {len(b.hcs):4d} HCS   named fields: {named:3d}"
          f"   sides: {','.join(SIDES[n])}   |  {len(b.ins)} interneurons")
B["male-cns"].hcs.type.value_counts().rename("n cells").to_frame().T

male-cns   339 HCS   named fields: 191   sides: L,R   |  452 interneurons
manc       333 HCS   named fields: 333   sides: L,R   |  555 interneurons
fanc       123 HCS   named fields:   0   sides: L   |  693 interneurons


type,SApp,"SApp09,SApp22",SApp08,"SApp06,SApp15",SApp01,SApp05,"SApp02,SApp03","SNpp34,SApp16","SNpp20,SApp02",SApp07
n cells,148,74,47,37,19,3,3,3,3,2


## 2 · Sanity gate

Direct HCS input as a share of each motor neuron's total input.  This must
reproduce across datasets before anything else is trusted — if it drifts, the
usual culprit is the side field (MANC uses `LHS`/`RHS`, male-cns uses `L`/`R`).

This table is the foundation the whole analysis rests on, since direct connections
are all that is computed.

In [5]:
def direct_pct(b):
    d = (b.direct
         .merge(b.hcs.rename(columns={"bodyId":"s","side":"s_side"})[["s","s_side"]], on="s")
         .merge(b.mns.rename(columns={"bodyId":"d","side":"d_side"})[["d","d_side","type","post"]], on="d"))
    d = d[d.s_side == d.d_side]                      # input is ~98% ipsilateral
    return (d.groupby(["type","d_side"])
              .apply(lambda x: pd.Series({"n_hcs": x.s.nunique(), "syn": x.w.sum(),
                                          "pct_of_input": round(100*x.w.sum()/x.post.iloc[0], 2)}),
                     include_groups=False))

for n, b in B.items():
    print(f"\n--- {n} ({b.dataset}) ---"); print(direct_pct(b).to_string())


--- male-cns (male-cns:v1.0) ---
               n_hcs    syn  pct_of_input
type   d_side                            
b1 MN  L        27.0  481.0          3.64
       R        28.0  397.0          3.33
b2 MN  L        34.0  300.0          1.93
       R        32.0  240.0          1.75
tp1 MN L        12.0  115.0          1.27
       R        11.0  102.0          1.22
tp2 MN R         1.0    4.0          0.04
tpn MN L         6.0   71.0          0.69
       R        10.0   51.0          0.62

--- manc (manc:v1.2.3) ---
               n_hcs    syn  pct_of_input
type   d_side                            
b1 MN  L        29.0  640.0          4.32
       R        32.0  766.0          4.79
b2 MN  L        41.0  506.0          2.76
       R        39.0  532.0          3.11
tp1 MN L        20.0  185.0          1.41
       R        13.0  177.0          1.58
tp2 MN L         2.0   11.0          0.08
       R         2.0    7.0          0.06
tpn MN L         7.0   65.0          0.54
       R      

**Read this before going further.**

In both male datasets the ordering is `b1 > b2 > tp1 > tpn >> tp2`, and **tp2 receives
essentially no direct HCS input at all** — 1 synapse on the left in male-cns.  That
bounds what this analysis can say: a comparison built on direct connections reports
"tp2 resembles nothing", which is a statement about the measurement rather than about
tp2.  Section 6 therefore draws no conclusion about tp2.

**FANC orders the two weak inputs differently**: `b1 > b3 > b2 > tpn > tp1 >> tp2`,
putting tpn (1.02%) marginally above tp1 (0.84%).  Two things to note before reading
anything into that. It is a difference of 9 afferents against 5, on a partial
reconstruction of one hemisphere. And more importantly it is a statement about *how
much* haltere input each receives, **not about which sensilla they share with b1/b2** —
which is what this project actually measures. Section 6 shows the similarity ordering is
unaffected.

## 3 · Controls

These decide whether the numbers in Section 6 mean anything, so they come first.

**Positive control.** Individual HCS and interneurons cannot be matched between
hemispheres, so left and right motor neurons share no cell-level feature.  But
interneuron `group` *is* bilateral, so re-expressing the premotor matrix over
`(ipsi/contra, group)` gives a space both sides live in.  Each motor neuron's left
and right copy must then come out as mutual nearest neighbours.  If this fails, the
normalisation is wrong and nothing downstream is interpretable.

The premotor matrix is computed for this control alone; no figure uses it.

**FANC supports neither L/R control.** Only one hemisphere was reconstructed, so there
is no second measurement to check against. That is a limitation of the reconstruction,
not of the method — and it is why FANC is treated as corroboration of a result
established on the two-hemisphere datasets rather than as independent proof.

In [7]:
def controls(b):
    sides = H.DATASETS[b.name]["sides"]
    D = H.direct_matrix(b)
    sims = {sd: H.similarity(H.prep(H.side(D, sd))) for sd in sides}
    if len(sides) < 2:
        return D, sims, {}, None
    Pg  = H.collapse_side_relative(H.premotor_matrix(b), b.ins, "group")
    S10 = H.similarity(H.prep(Pg))
    nn  = {c: S10[c].drop(c).idxmax() for c in S10.columns}
    ok  = all(v == f"{c.rsplit('_',1)[0]}_{'R' if c.endswith('_L') else 'L'}"
              for c, v in nn.items())
    return D, sims, nn, ok

D, S, NN, OK = {}, {}, {}, {}
for n, b in B.items():
    D[n], S[n], NN[n], OK[n] = controls(b)
    if OK[n] is None:
        print(f"{n:9s} single hemisphere — L/R controls not applicable"); continue
    print(f"{n:9s} L/R mutual-nearest-neighbour control: {'PASS' if OK[n] else 'FAIL'}")
    print("   " + "  ".join(f"{c}->{v}" for c, v in NN[n].items()))

male-cns  L/R mutual-nearest-neighbour control: PASS
   b1_L->b1_R  b1_R->b1_L  b2_L->b2_R  b2_R->b2_L  tp1_L->tp1_R  tp1_R->tp1_L  tp2_L->tp2_R  tp2_R->tp2_L  tpn_L->tpn_R  tpn_R->tpn_L
manc      L/R mutual-nearest-neighbour control: PASS
   b1_L->b1_R  b1_R->b1_L  b2_L->b2_R  b2_R->b2_L  tp1_L->tp1_R  tp1_R->tp1_L  tp2_L->tp2_R  tp2_R->tp2_L  tpn_L->tpn_R  tpn_R->tpn_L
fanc      single hemisphere — L/R controls not applicable


**Reliability.** Where two hemispheres exist they are independent measurements
(disjoint features), so the correlation between the left and the right matrix estimates
reproducibility.

Note the `n` column: where a motor neuron has no input its similarity is
**undefined, not zero**, and those pairs are dropped rather than imputed. tp2 in
male-cns is exactly that case.

In [9]:
rows = []
for n in B:
    if len(SIDES[n]) < 2:
        rows.append({"dataset": n, "L/R reliability r": np.nan, "n pairs": 0,
                     "n pairs possible": len(H.offdiag(S[n]["L"]))}); continue
    L = H.offdiag(S[n]["L"]).sim.values
    R = H.offdiag(S[n]["R"]).sim.values
    m = np.isfinite(L) & np.isfinite(R)
    rows.append({"dataset": n, "L/R reliability r": round(pearsonr(L[m], R[m])[0], 3),
                 "n pairs": int(m.sum()), "n pairs possible": len(L)})
pd.DataFrame(rows)

,dataset,L/R reliability r,n pairs,n pairs possible
0,male-cns,0.986,6,10
1,manc,0.979,10,10
2,fanc,NaN,0,10


## 4 · The analysis

Every edge is divided by the **target's** total postsynaptic count, so the result is
an input *fraction*; without that, motor neurons would rank by size (b2 has ~15.6k
input sites against tp1's ~9k).  Edges below `MIN_SYN = 3` synapses are dropped.

| level | features | question |
|---|---|---|
| **direct** | HCS cells | do two MNs read the same sensilla directly? |

Each motor neuron's column is sqrt-compressed and scaled to unit length before the
cosine.  The sqrt keeps one dominant sensillum from deciding the whole score; the
per-column scaling is why adding b3 as a sixth column cannot move any of the five
existing values.

One caveat specific to comparing datasets: the *denominator* is not computed identically
everywhere — MANC is more completely proofread and FANC's total counts synapses from
unproofread fragments. That affects the input-map shading, not the similarities, because
each column is unit-scaled on its own.

In [11]:
for n in B:
    print(f"\n===== {n} — cosine similarity, left hemisphere =====")
    print(S[n]["L"].round(2).to_string())
print("\n(figures for every dataset are written by run_hcs_similarity.py)")


===== male-cns — cosine similarity, left hemisphere =====
       b1_L  b2_L  tp1_L  tp2_L  tpn_L
b1_L   1.00  0.72   0.62    NaN    0.0
b2_L   0.72  1.00   0.32    NaN    0.0
tp1_L  0.62  0.32   1.00    NaN    0.0
tp2_L   NaN   NaN    NaN    NaN    NaN
tpn_L  0.00  0.00   0.00    NaN    1.0

===== manc — cosine similarity, left hemisphere =====
       b1_L  b2_L  tp1_L  tp2_L  tpn_L
b1_L   1.00  0.77   0.63   0.00   0.00
b2_L   0.77  1.00   0.37   0.00   0.00
tp1_L  0.63  0.37   1.00   0.00   0.06
tp2_L  0.00  0.00   0.00   1.00   0.27
tpn_L  0.00  0.00   0.06   0.27   1.00

===== fanc — cosine similarity, left hemisphere =====
       b1_L  b2_L  tp1_L  tp2_L  tpn_L
b1_L   1.00  0.74   0.45    NaN    0.0
b2_L   0.74  1.00   0.25    NaN    0.0
tp1_L  0.45  0.25   1.00    NaN    0.0
tp2_L   NaN   NaN    NaN    NaN    NaN
tpn_L  0.00  0.00   0.00    NaN    1.0

(figures for every dataset are written by run_hcs_similarity.py)


## 5 · Cross-dataset concordance

male-cns and MANC are **two different flies**, independently reconstructed — their
body IDs do not overlap, and `mancBodyid` exists precisely because the volumes are
separate. Agreement between them speaks to whether the structure generalises across
individuals, not just whether the pipeline is self-consistent.

First confirm the ten motor neurons are genuinely the same cells across those two
datasets rather than trusting the type strings.  FANC is a different animal with its own
identifier space and no correspondence table, so it cannot be checked this way — its
motor neurons are matched by the collaborator's labels, two of which (`b3_u`, `dtpmn_u`
→ b3, tp1) are flagged uncertain upstream.

In [13]:
XREF = {("b1 MN","L"):(801310,10013), ("b1 MN","R"):(804301,10017),
        ("b2 MN","L"):(801137,10131), ("b2 MN","R"):(801350,10064),
        ("tp1 MN","L"):(802315,10521), ("tp1 MN","R"):(802028,10778),
        ("tp2 MN","L"):(801949,10270), ("tp2 MN","R"):(800899,10766),
        ("tpn MN","L"):(801185,10543), ("tpn MN","R"):(801815,11294)}
mc = B["male-cns"].mns.set_index(["type","side"]).bodyId
mn = B["manc"].mns.set_index(["type","side"]).bodyId
assert all(mc[k] == v[0] and mn[k] == v[1] for k, v in XREF.items())
print("all 10 motor neurons map 1:1 between male-cns and MANC")

all 10 motor neurons map 1:1 between male-cns and MANC


In [15]:
v = {n: np.nanmean([H.offdiag(S[n][sd]).sim.values for sd in SIDES[n]], axis=0)
     for n in B if len(SIDES[n]) == 2}
m = np.isfinite(v["male-cns"]) & np.isfinite(v["manc"])
print(f"direct  concordance r = {pearsonr(v['male-cns'][m], v['manc'][m])[0]:.3f}"
      f"   (n = {int(m.sum())} pairs with both defined)")

direct  concordance r = 0.927   (n = 10 pairs with both defined)


> **Caveat.** Most pairs sit at or near zero in male-cns, so this correlation is
> carried by only two or three informative points — b1–b2, b1–tp1, b2–tp1. It says
> those three agree across animals; it does not validate the near-zero pairs.

---
# 6 · Interpretation — the un-blinded reading

Everything above treated the five motor neurons identically. **Only now** do we
split them into the well-characterised reference (b1, b2) and the understudied
queries (tp1, tp2, tpn), and ask which query is closest to the reference.

The robustness check is across **3 metrics × 3 datasets**, one of which is a different
sex: the answer should stand or fall on whether it survives every combination rather
than on a favourable one.

In [17]:
REF, QRY = ["b1 MN","b2 MN"], ["tp1 MN","tp2 MN","tpn MN"]

def to_ref(n, metric="cosine"):
    out = {}
    for q in QRY:
        vals = []
        for sd in SIDES[n]:
            Sm = H.similarity(H.prep(H.side(D[n], sd)), metric)
            qs = f"{q.replace(' MN','')}_{sd}"
            vals += [Sm.loc[qs, f"{r.replace(' MN','')}_{sd}"] for r in REF]
        vals = [x for x in vals if np.isfinite(x)]
        out[q.replace(" MN","")] = np.mean(vals) if vals else np.nan
    return pd.Series(out, name=metric)

for n in B:
    print(f"\n===== {n}: mean similarity to b1 and b2 =====")
    print(pd.DataFrame({m: to_ref(n, m) for m in H.METRICS}).round(3).to_string())


===== male-cns: mean similarity to b1 and b2 =====
     cosine  pearson  jaccard
tp1   0.418    0.388    0.191
tp2   0.000   -0.016    0.000
tpn   0.000   -0.044    0.000

===== manc: mean similarity to b1 and b2 =====
     cosine  pearson  jaccard
tp1   0.499    0.468    0.263
tp2   0.000   -0.025    0.000
tpn   0.027   -0.028    0.010

===== fanc: mean similarity to b1 and b2 =====
     cosine  pearson  jaccard
tp1   0.351    0.297    0.123
tp2     NaN      NaN      NaN
tpn   0.000   -0.153    0.000


### Does the ranking survive every metric and every dataset?

`tp2` is undefined wherever it has no direct input, so the ranking is reported over the
queries that *are* defined, with the count shown.  Dropping an undefined value is the
only honest option — ranking it as though it were zero would assert a measurement that
was never made.

In [19]:
tally = []
for n in B:
    for metric in H.METRICS:
        col = to_ref(n, metric).dropna()
        if len(col) < 2:
            continue
        tally.append({"dataset": n, "metric": metric, "n_defined": len(col),
                      "ranking": " > ".join(col.sort_values(ascending=False).index)})
tally = pd.DataFrame(tally)
print(f"{len(tally)} independent comparisons\n")
print(tally.ranking.value_counts().rename("count").to_frame().to_string())
first = tally.ranking.str.split(" > ").str[0].value_counts()
print(f"\nranked closest to b1/b2:\n{first.to_string()}")
print(f"\nper dataset:\n{tally.groupby('dataset').ranking.apply(lambda x: '; '.join(sorted(set(x)))).to_string()}")

9 independent comparisons

                 count
ranking               
tp1 > tp2 > tpn      4
tp1 > tpn            3
tp1 > tpn > tp2      2

ranked closest to b1/b2:
ranking
tp1    9

per dataset:
dataset
fanc                               tp1 > tpn
male-cns                     tp1 > tp2 > tpn
manc        tp1 > tp2 > tpn; tp1 > tpn > tp2


## What this shows

**tp1 is the tp motor neuron whose direct haltere input most resembles b1 and b2**, in
every metric and in all three connectomes — including a female. It leads by a wide
margin, not marginally. On the biological reading this makes tp1 the strongest candidate
for being recruited under haltere-driven conditions similar to those that drive b1 and b2.

**The core similarity block reproduces across three animals and both sexes:**

| | male-CNS | MANC | FANC |
|---|---|---|---|
| b1–b2 | 0.72 | 0.77 | 0.74 |
| b1–tp1 | 0.56 | 0.64 | 0.45 |
| b2–tp1 | 0.27 | 0.36 | 0.25 |
| b3–b1 / b3–b2 | ~0.00 | ~0.00 | 0.00 / 0.00 |

That b1–b2 lands within 0.05 across three independent reconstructions is the strongest
evidence here that the measurement is picking up real anatomy rather than pipeline
artefact.

**tpn is clearly further from the reference.** It takes direct haltere input — in FANC
slightly more of it than tp1 does — but from a largely different set of sensilla, so its
similarity to b1/b2 stays at or near zero everywhere. This is the distinction the
Section 2 note flagged: **input volume and input identity are different questions, and
they disagree here.** FANC gives tpn more haltere synapses than tp1 while still placing
tp1 far closer to b1/b2. Reporting only the synapse counts would invert the conclusion.

**This analysis says nothing about tp2**, and that is a limit rather than a result. tp2
receives essentially no direct haltere input in any of the three datasets — one synapse
in male-cns, and in FANC three contacts of two synapses each, all below the `MIN_SYN = 3`
threshold. Do not read its near-zero or `n/a` values as "tp2 shares nothing with b1/b2";
read them as "tp2 has no direct haltere input to compare". Answering the tp2 question
would require a polysynaptic analysis, which this pipeline does not do.

**b3 is the clearest negative result, and it now replicates three times.** It takes as
much direct haltere input as b1 — second most in FANC — yet its similarity to b1 and b2
is ~0.00 in every connectome: it reads a large, almost completely disjoint population of
sensilla. `hcs_to_b1b2b3.csv` gives the cell-level evidence for the two neuPrint
datasets, and the b3 figures show the two blocks side by side.

## Caveats to carry into any writeup

- **Direct connections only.** Anything routed through an interneuron is invisible
  here. That is what makes the analysis silent on tp2.
- **~45% of HCS carry no field assignment** in the neuPrint datasets (generic `SApp`),
  and FANC carries none at all. The analysis can show two motor neurons sample
  overlapping sensilla, but often not *which* field — so the yaw/pitch/roll mapping
  stays indirect.
- **n = 3 animals: two male, one female.** Left/right is a within-animal check, not a
  replicate, and FANC contributes only one hemisphere, so it has no internal control.
- **FANC covers 123 afferents against ~340**, one hemisphere only. Its *absence* of a
  connection is weaker evidence than its presence.
- **FANC's b3 and tp1 identifications are flagged uncertain upstream** (`b3_u`,
  `dtpmn_u`) — exactly the two motor neurons the conclusions lean on hardest. The b3
  result does not depend on FANC, but treat FANC's tp1 value as corroboration rather
  than as an independent measurement.
- **Input denominators differ between connectomes.** MANC percentages run higher than
  male-cns (denser proofreading) and FANC's counts unproofread fragments. Compare rank
  orders and band structure, not absolute values. The similarity figures are immune,
  since each column is unit-scaled independently.
- **Undefined ≠ zero.** Where a motor neuron has no input, similarity is NaN and those
  pairs are dropped from correlations. Imputing zero would manufacture agreement.
  How many hemispheres back each cell is recorded in
  `similarity_1hop_n_hemispheres.csv`.
- Thresholds (`MIN_SYN = 3`) and the sqrt compression are choices; the three metrics
  are reported so their influence is visible rather than hidden.